# 11_Modeling Datasets And Training

This notebook builds the modeling-facing artifacts from the completed full-project cohort and aligned sequence outputs. It creates a static event-level feature table, deterministic patient-level splits, sequence manifests, and outcome-integration hooks for the project models.


## 2. Scope

This notebook is the handoff from extraction to modeling. It assumes notebook 10 has already produced:

- `full_project_candidate_cohort.csv`
- event-aligned sequence datasets for HbA1c, core labs, glucose-lowering medications, and body-size

Important current caveat:

- the full-project outcome label file is not yet present under `data/full_project/outcomes/`
- so this notebook builds predictor-ready and split-ready artifacts now, and will only run training if an outcome file is added later


In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import numpy as np

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
FULL_PROJECT_DIR = DATA_DIR / 'full_project'
FULL_COHORT_DIR = FULL_PROJECT_DIR / 'cohorts'
SEQUENCE_ALIGNMENT_DIR = FULL_PROJECT_DIR / 'sequence_datasets' / 'event_aligned'
MODELING_DIR = FULL_PROJECT_DIR / 'modeling'
MODELING_DIR.mkdir(parents=True, exist_ok=True)

STATIC_EVENT_TABLE_PATH = MODELING_DIR / 'static_event_modeling_table.csv'
PATIENT_SPLITS_PATH = MODELING_DIR / 'patient_splits.csv'
EVENT_SPLITS_PATH = MODELING_DIR / 'event_splits.csv'
STATIC_FEATURE_SUMMARY_PATH = MODELING_DIR / 'static_feature_summary.csv'
SEQUENCE_DOMAIN_SUMMARY_PATH = MODELING_DIR / 'sequence_domain_summary.csv'
MODELING_MANIFEST_PATH = MODELING_DIR / 'modeling_manifest.json'

FULL_CANDIDATE_COHORT_PATH = FULL_COHORT_DIR / 'full_project_candidate_cohort.csv'
FULL_CANDIDATE_ATTRITION_PATH = FULL_COHORT_DIR / 'full_project_candidate_attrition.csv'

ALIGNED_SEQUENCE_PATHS = {
    'sequence_hba1c': SEQUENCE_ALIGNMENT_DIR / 'sequence_hba1c_event_aligned.csv',
    'sequence_core_labs': SEQUENCE_ALIGNMENT_DIR / 'sequence_core_labs_event_aligned.csv',
    'sequence_glucose_meds': SEQUENCE_ALIGNMENT_DIR / 'sequence_glucose_meds_event_aligned.csv',
    'sequence_body_size': SEQUENCE_ALIGNMENT_DIR / 'sequence_body_size_event_aligned.csv',
}

OUTCOME_DIR = FULL_PROJECT_DIR / 'outcomes'
OUTCOME_CANDIDATE_PATHS = [
    OUTCOME_DIR / 'full_project_outcome_hba1c.csv',
    OUTCOME_DIR / 'full_project_outcome_labels.csv',
    OUTCOME_DIR / 'full_project_followup_hba1c.csv',
]

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## 3. Load Final Cohort And Aligned Sequence Files


In [2]:
full_project_candidate_cohort = pd.read_csv(
    FULL_CANDIDATE_COHORT_PATH,
    parse_dates=['index_date', 'first_t2d_date', 'baseline_hba1c_date']
) if FULL_CANDIDATE_COHORT_PATH.exists() else pd.DataFrame()

full_project_candidate_attrition = pd.read_csv(FULL_CANDIDATE_ATTRITION_PATH) if FULL_CANDIDATE_ATTRITION_PATH.exists() else pd.DataFrame()

sequence_file_summary = []
for domain, path in ALIGNED_SEQUENCE_PATHS.items():
    sequence_file_summary.append({
        'domain': domain,
        'exists': path.exists(),
        'size_mb': round(path.stat().st_size / (1024 * 1024), 2) if path.exists() else 0.0,
        'path': str(path),
    })
sequence_file_summary = pd.DataFrame(sequence_file_summary)

print('Candidate cohort rows:', len(full_project_candidate_cohort))
print('Candidate cohort patients:', full_project_candidate_cohort['patient_id'].nunique() if not full_project_candidate_cohort.empty else 0)
display(sequence_file_summary)
if not full_project_candidate_attrition.empty:
    display(full_project_candidate_attrition)


Candidate cohort rows: 120224
Candidate cohort patients: 94651


,domain,exists,size_mb,path
0,sequence_hba1c,True,39.77,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
1,sequence_core_labs,True,326.94,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
2,sequence_glucose_meds,True,821.47,/uufs/chpc.utah.edu/common/home/kukhareva-grou...
3,sequence_body_size,True,2305.36,/uufs/chpc.utah.edu/common/home/kukhareva-grou...


,step,rows,patients
0,initial_candidate_initiations,3024783,2139317
1,after_demographics_age_filter,496358,352591
2,after_t2d_requirement,411692,297321
3,after_not_first_line_requirement,305577,227352
4,after_eskd_exclusion,295841,219917
5,after_insulin_overlap_exclusion,203782,158299
6,after_baseline_hba1c_requirement,120224,94651


## 4. Rebuild The Event Cohort Used For Modeling

The extraction notebook assigned event IDs deterministically by sorting `(patient_id, index_date, drug_class)`. We rebuild that same event lookup here so the static table and the aligned sequence datasets use the same event IDs.


In [3]:
if full_project_candidate_cohort.empty:
    event_lookup = pd.DataFrame(columns=['event_id', 'patient_id', 'drug_class', 'index_date'])
else:
    event_lookup = (
        full_project_candidate_cohort[['patient_id', 'drug_class', 'index_date']]
        .drop_duplicates()
        .sort_values(['patient_id', 'index_date', 'drug_class'])
        .reset_index(drop=True)
        .copy()
    )
    event_lookup['patient_id'] = event_lookup['patient_id'].astype('string')
    event_lookup['event_id'] = ['event_%07d' % (i + 1) for i in range(len(event_lookup))]

cohort_event_table = event_lookup.merge(
    full_project_candidate_cohort,
    on=['patient_id', 'drug_class', 'index_date'],
    how='left',
    validate='one_to_one'
) if not event_lookup.empty else pd.DataFrame()

if not cohort_event_table.empty:
    cohort_event_table['t2d_duration_days'] = (cohort_event_table['index_date'] - cohort_event_table['first_t2d_date']).dt.days

print('Modeling events:', len(cohort_event_table))
print('Modeling patients:', cohort_event_table['patient_id'].nunique() if not cohort_event_table.empty else 0)
display(cohort_event_table.head(10))


Modeling events: 120224
Modeling patients: 94651


,patient_id,drug_class,index_date,event_id,age_at_index,sex,first_t2d_date,baseline_hba1c_date,code_system,code,baseline_hba1c,units_of_measure,days_from_index,t2d_duration_days
0,#A##B,GLP1,2019-07-03,event_0000001,74.0,M,2019-03-28,2019-06-19,LOINC,17856-6,8.0,%,14,97
1,#A##B,SGLT2,2023-12-14,event_0000002,78.0,M,2019-03-28,2023-12-04,LOINC,17856-6,7.5,%,10,1722
2,#A#0B,GLP1,2023-08-16,event_0000003,53.0,F,2022-03-18,2023-08-12,LOINC,4548-4,7.2,%,4,516
3,#A#1B,SGLT2,2015-07-02,event_0000004,73.0,M,2014-10-13,2015-07-02,LOINC,4548-4,8.8,%,0,262
4,#A#2,DPP4,2018-08-02,event_0000005,47.0,F,2018-04-24,2018-08-02,LOINC,4548-4,8.2,%,0,100
5,#A#2,SGLT2,2018-08-16,event_0000006,47.0,F,2018-04-24,2018-08-02,LOINC,4548-4,8.2,%,14,114
6,#A#2,GLP1,2020-02-24,event_0000007,49.0,F,2018-04-24,2020-02-06,LOINC,4548-4,9.1,%,18,671
7,#A#9,DPP4,2008-03-10,event_0000008,67.0,M,2005-10-05,2007-11-17,LOINC,4548-4,7.3,%,114,887
8,#A#AB,GLP1,2022-02-09,event_0000009,29.0,F,2021-12-27,2022-02-01,LOINC,4548-4,9.1,%,8,44
9,#A#AD,GLP1,2025-11-19,event_0000010,66.0,M,2025-06-12,2025-09-26,LOINC,4548-4,8.5,%,54,160


## 5. Build Latest Pre-Index Core-Lab Snapshots

This section reads the aligned core-lab sequence file in chunks and keeps the closest pre-index value for each `(event_id, feature_name)` pair.


In [4]:
SNAPSHOT_CHUNKSIZE = 200_000


def reduce_latest_feature_rows(current_best: pd.DataFrame, chunk_best: pd.DataFrame) -> pd.DataFrame:
    if current_best.empty:
        combined = chunk_best.copy()
    else:
        combined = pd.concat([current_best, chunk_best], ignore_index=True)
    combined['days_before_index'] = pd.to_numeric(combined['days_before_index'], errors='coerce')
    combined['event_date'] = pd.to_datetime(combined['event_date'], errors='coerce')
    combined = combined.sort_values(['event_id', 'feature_name', 'days_before_index', 'event_date'])
    return combined.groupby(['event_id', 'feature_name'], as_index=False).first()


def build_latest_snapshot(path: Path, value_col: str, allowed_features: list[str] | None = None, chunksize: int = SNAPSHOT_CHUNKSIZE) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame(columns=['event_id', 'feature_name', value_col, 'days_before_index', 'event_date', 'units_of_measure'])

    best = pd.DataFrame(columns=['event_id', 'feature_name', value_col, 'days_before_index', 'event_date', 'units_of_measure'])
    usecols = ['event_id', 'feature_name', value_col, 'days_before_index', 'event_date', 'units_of_measure']

    for chunk in pd.read_csv(path, usecols=lambda c: c in set(usecols), chunksize=chunksize):
        if chunk.empty:
            continue
        chunk['event_id'] = chunk['event_id'].astype('string')
        chunk['feature_name'] = chunk['feature_name'].astype('string')
        chunk['days_before_index'] = pd.to_numeric(chunk['days_before_index'], errors='coerce')
        chunk['event_date'] = pd.to_datetime(chunk['event_date'], errors='coerce')
        chunk[value_col] = pd.to_numeric(chunk[value_col], errors='coerce')
        chunk = chunk.dropna(subset=['event_id', 'feature_name', value_col, 'days_before_index'])
        if allowed_features is not None:
            chunk = chunk[chunk['feature_name'].isin(allowed_features)].copy()
        if chunk.empty:
            continue
        chunk_best = (
            chunk.sort_values(['event_id', 'feature_name', 'days_before_index', 'event_date'])
            .groupby(['event_id', 'feature_name'], as_index=False)
            .first()
        )
        best = reduce_latest_feature_rows(best, chunk_best)

    return best


core_lab_snapshot_long = build_latest_snapshot(
    ALIGNED_SEQUENCE_PATHS['sequence_core_labs'],
    value_col='lab_result_num_val',
    allowed_features=['egfr', 'hdl', 'total_cholesterol', 'alt'],
)

display(core_lab_snapshot_long.head(12))
print('Core-lab snapshot rows:', len(core_lab_snapshot_long))


,event_id,feature_name,event_date,days_before_index,lab_result_num_val,units_of_measure
0,event_0000001,alt,2019-06-19,14,33.00,U/L
1,event_0000001,egfr,2019-06-19,14,45.00,mL/min/{1.73_m2}
2,event_0000001,hdl,2019-06-19,14,42.00,mg/dL
3,event_0000001,total_cholesterol,2019-06-19,14,141.00,mg/dL
4,event_0000002,alt,2023-12-04,10,18.00,U/L
5,event_0000002,egfr,2023-12-04,10,22.00,mL/min/{1.73_m2}
6,event_0000002,hdl,2023-12-04,10,63.00,mg/dL
7,event_0000002,total_cholesterol,2023-12-04,10,145.00,mg/dL
8,event_0000003,alt,2023-08-12,4,140.00,U/L
9,event_0000003,egfr,2023-08-12,4,100.00,mL/min/{1.73_m2}


Core-lab snapshot rows: 403578


## 6. Build Latest Body-Size Snapshots And Final BMI


In [5]:
body_snapshot_long = build_latest_snapshot(
    ALIGNED_SEQUENCE_PATHS['sequence_body_size'],
    value_col='value',
    allowed_features=['bmi_body_size', 'weight_body_size', 'height_body_size'],
)

def pivot_snapshot(snapshot_long: pd.DataFrame, value_col: str, prefix: str = '') -> pd.DataFrame:
    if snapshot_long.empty:
        return pd.DataFrame(columns=['event_id'])
    wide = snapshot_long.pivot(index='event_id', columns='feature_name', values=value_col).reset_index()
    wide.columns.name = None
    if prefix:
        rename_map = {c: f'{prefix}{c}' for c in wide.columns if c != 'event_id'}
        wide = wide.rename(columns=rename_map)
    return wide

core_lab_snapshot = pivot_snapshot(core_lab_snapshot_long, 'lab_result_num_val')
body_snapshot = pivot_snapshot(body_snapshot_long, 'value')

if not body_snapshot.empty:
    body_snapshot['bmi_direct'] = pd.to_numeric(body_snapshot.get('bmi_body_size'), errors='coerce')
    body_snapshot['weight_lb'] = pd.to_numeric(body_snapshot.get('weight_body_size'), errors='coerce')
    body_snapshot['height_in'] = pd.to_numeric(body_snapshot.get('height_body_size'), errors='coerce')
    body_snapshot['bmi_derived'] = np.where(
        body_snapshot['weight_lb'].notna() & body_snapshot['height_in'].notna() & (body_snapshot['height_in'] > 0),
        703.0 * body_snapshot['weight_lb'] / (body_snapshot['height_in'] ** 2),
        np.nan,
    )
    body_snapshot['bmi_final'] = body_snapshot['bmi_direct'].fillna(body_snapshot['bmi_derived'])

display(body_snapshot.head(12))
print('Body-size snapshot rows:', len(body_snapshot))


,event_id,bmi_body_size,height_body_size,weight_body_size,bmi_direct,weight_lb,height_in,bmi_derived,bmi_final
0,event_0000001,NaN,67.0,NaN,NaN,NaN,67.0,NaN,NaN
1,event_0000002,NaN,66.0,NaN,NaN,NaN,66.0,NaN,NaN
2,event_0000003,NaN,62.0,NaN,NaN,NaN,62.0,NaN,NaN
3,event_0000004,NaN,65.0,NaN,NaN,NaN,65.0,NaN,NaN
4,event_0000005,NaN,62.5,NaN,NaN,NaN,62.5,NaN,NaN
5,event_0000006,NaN,62.5,NaN,NaN,NaN,62.5,NaN,NaN
6,event_0000007,NaN,61.0,NaN,NaN,NaN,61.0,NaN,NaN
7,event_0000008,NaN,65.0,NaN,NaN,NaN,65.0,NaN,NaN
8,event_0000009,NaN,65.0,NaN,NaN,NaN,65.0,NaN,NaN
9,event_0000010,NaN,67.0,NaN,NaN,NaN,67.0,NaN,NaN


Body-size snapshot rows: 107393


## 7. Build Static Pre-Index Glucose-Medication History Features

This section summarizes the aligned glucose-medication sequence into event-level baseline medication-history features that are useful for the static models.


In [6]:
MED_SUMMARY_CHUNKSIZE = 200_000


def init_med_summary_template() -> pd.DataFrame:
    return pd.DataFrame(columns=[
        'event_id',
        'glucose_med_record_rows_2y',
        'first_glucose_med_days_before_index',
        'last_glucose_med_days_before_index',
        'prior_metformin_2y',
        'prior_insulin_2y',
        'prior_glp1_2y',
        'prior_sglt2_2y',
        'prior_dpp4_2y',
        'prior_sulfonylurea_2y',
        'prior_tzd_2y',
    ])


def combine_med_summaries(current: pd.DataFrame, chunk_summary: pd.DataFrame) -> pd.DataFrame:
    if current.empty:
        combined = chunk_summary.copy()
    else:
        combined = pd.concat([current, chunk_summary], ignore_index=True)
    agg = {
        'glucose_med_record_rows_2y': 'sum',
        'first_glucose_med_days_before_index': 'max',
        'last_glucose_med_days_before_index': 'min',
        'prior_metformin_2y': 'max',
        'prior_insulin_2y': 'max',
        'prior_glp1_2y': 'max',
        'prior_sglt2_2y': 'max',
        'prior_dpp4_2y': 'max',
        'prior_sulfonylurea_2y': 'max',
        'prior_tzd_2y': 'max',
    }
    return combined.groupby('event_id', as_index=False).agg(agg)


def build_medication_summary(path: Path, chunksize: int = MED_SUMMARY_CHUNKSIZE) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return init_med_summary_template()

    summary = init_med_summary_template()
    usecols = ['event_id', 'days_before_index', 'code', 'code_description', 'path', 'event_date']

    for chunk in pd.read_csv(path, usecols=lambda c: c in set(usecols), chunksize=chunksize):
        if chunk.empty:
            continue
        chunk['event_id'] = chunk['event_id'].astype('string')
        chunk['days_before_index'] = pd.to_numeric(chunk['days_before_index'], errors='coerce')
        chunk['event_date'] = pd.to_datetime(chunk['event_date'], errors='coerce')
        chunk = chunk.dropna(subset=['event_id', 'days_before_index'])
        if chunk.empty:
            continue

        desc = (
            chunk.get('code_description', pd.Series('', index=chunk.index)).fillna('').astype('string')
            + ' ' +
            chunk.get('path', pd.Series('', index=chunk.index)).fillna('').astype('string')
        ).str.lower()

        chunk = chunk.assign(
            med_record=1,
            prior_metformin_2y=desc.str.contains('metformin', regex=False).astype(int),
            prior_insulin_2y=(desc.str.contains('insulin', regex=False) | desc.str.contains('/a10a/', regex=False)).astype(int),
            prior_glp1_2y=(desc.str.contains('glutide', regex=False) | desc.str.contains('/a10bj/', regex=False)).astype(int),
            prior_sglt2_2y=(desc.str.contains('gliflozin', regex=False) | desc.str.contains('/a10bk/', regex=False)).astype(int),
            prior_dpp4_2y=(desc.str.contains('gliptin', regex=False) | desc.str.contains('/a10bh/', regex=False)).astype(int),
            prior_sulfonylurea_2y=(
                desc.str.contains('/a10bb/', regex=False)
                | desc.str.contains('gliclazide', regex=False)
                | desc.str.contains('glimepiride', regex=False)
                | desc.str.contains('glipizide', regex=False)
                | desc.str.contains('glyburide', regex=False)
            ).astype(int),
            prior_tzd_2y=(
                desc.str.contains('/a10bg/', regex=False)
                | desc.str.contains('pioglitazone', regex=False)
                | desc.str.contains('rosiglitazone', regex=False)
            ).astype(int),
        )

        chunk_summary = chunk.groupby('event_id', as_index=False).agg({
            'med_record': 'sum',
            'days_before_index': ['max', 'min'],
            'prior_metformin_2y': 'max',
            'prior_insulin_2y': 'max',
            'prior_glp1_2y': 'max',
            'prior_sglt2_2y': 'max',
            'prior_dpp4_2y': 'max',
            'prior_sulfonylurea_2y': 'max',
            'prior_tzd_2y': 'max',
        })
        chunk_summary.columns = [
            'event_id',
            'glucose_med_record_rows_2y',
            'first_glucose_med_days_before_index',
            'last_glucose_med_days_before_index',
            'prior_metformin_2y',
            'prior_insulin_2y',
            'prior_glp1_2y',
            'prior_sglt2_2y',
            'prior_dpp4_2y',
            'prior_sulfonylurea_2y',
            'prior_tzd_2y',
        ]
        summary = combine_med_summaries(summary, chunk_summary)

    return summary


glucose_med_summary = build_medication_summary(ALIGNED_SEQUENCE_PATHS['sequence_glucose_meds'])

display(glucose_med_summary.head(12))
print('Medication-summary rows:', len(glucose_med_summary))


,event_id,glucose_med_record_rows_2y,first_glucose_med_days_before_index,last_glucose_med_days_before_index,prior_metformin_2y,prior_insulin_2y,prior_glp1_2y,prior_sglt2_2y,prior_dpp4_2y,prior_sulfonylurea_2y,prior_tzd_2y
0,event_0000001,2,5,5,1,0,0,0,0,0,0
1,event_0000002,6,723,108,0,0,1,0,0,0,0
2,event_0000003,42,516,160,1,0,0,1,0,1,0
3,event_0000004,12,160,160,0,1,0,0,0,0,0
4,event_0000005,2,91,91,1,0,0,0,0,0,0
5,event_0000006,14,105,14,1,0,0,0,1,0,0
6,event_0000007,58,662,18,1,0,0,1,1,0,0
7,event_0000008,14,670,56,1,0,0,0,0,1,1
8,event_0000009,4,44,15,1,0,0,0,0,0,0
9,event_0000010,792,153,41,0,1,0,1,0,0,0


Medication-summary rows: 112862


## 8. Assemble The Static Event-Level Modeling Table


In [7]:
static_event_modeling_table = cohort_event_table.copy()

if not core_lab_snapshot.empty:
    static_event_modeling_table = static_event_modeling_table.merge(core_lab_snapshot, on='event_id', how='left')
if not body_snapshot.empty:
    static_event_modeling_table = static_event_modeling_table.merge(body_snapshot, on='event_id', how='left')
if not glucose_med_summary.empty:
    static_event_modeling_table = static_event_modeling_table.merge(glucose_med_summary, on='event_id', how='left')

for col in [
    'prior_metformin_2y',
    'prior_insulin_2y',
    'prior_glp1_2y',
    'prior_sglt2_2y',
    'prior_dpp4_2y',
    'prior_sulfonylurea_2y',
    'prior_tzd_2y',
]:
    if col in static_event_modeling_table.columns:
        static_event_modeling_table[col] = static_event_modeling_table[col].fillna(0).astype(int)

for col in ['baseline_hba1c', 'egfr', 'hdl', 'total_cholesterol', 'alt', 'bmi_final', 'weight_lb', 'height_in']:
    if col in static_event_modeling_table.columns:
        static_event_modeling_table[f'{col}_missing'] = static_event_modeling_table[col].isna().astype(int)

static_event_modeling_table.to_csv(STATIC_EVENT_TABLE_PATH, index=False)

static_feature_summary = pd.DataFrame([
    {
        'feature': col,
        'nonmissing_rows': int(static_event_modeling_table[col].notna().sum()),
        'nonmissing_pct': round(100.0 * static_event_modeling_table[col].notna().mean(), 2),
    }
    for col in ['age_at_index', 'sex', 'baseline_hba1c', 'egfr', 'hdl', 'total_cholesterol', 'alt', 'bmi_final', 'weight_lb', 'height_in']
    if col in static_event_modeling_table.columns
])
static_feature_summary.to_csv(STATIC_FEATURE_SUMMARY_PATH, index=False)

print(STATIC_EVENT_TABLE_PATH)
print('Static modeling rows:', len(static_event_modeling_table))
print('Static modeling patients:', static_event_modeling_table['patient_id'].nunique())
display(static_feature_summary)
display(static_event_modeling_table.head(10))


/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/modeling/static_event_modeling_table.csv
Static modeling rows: 120224
Static modeling patients: 94651


,feature,nonmissing_rows,nonmissing_pct
0,age_at_index,120224,100.00
1,sex,120214,99.99
2,baseline_hba1c,120224,100.00
3,egfr,114924,95.59
4,hdl,105987,88.16
5,total_cholesterol,106397,88.50
6,alt,76270,63.44
7,bmi_final,90052,74.90
8,weight_lb,88223,73.38
9,height_in,100036,83.21


,patient_id,drug_class,index_date,event_id,age_at_index,sex,first_t2d_date,baseline_hba1c_date,code_system,code,baseline_hba1c,units_of_measure,days_from_index,t2d_duration_days,alt,egfr,hdl,total_cholesterol,bmi_body_size,height_body_size,weight_body_size,bmi_direct,weight_lb,height_in,bmi_derived,bmi_final,glucose_med_record_rows_2y,first_glucose_med_days_before_index,last_glucose_med_days_before_index,prior_metformin_2y,prior_insulin_2y,prior_glp1_2y,prior_sglt2_2y,prior_dpp4_2y,prior_sulfonylurea_2y,prior_tzd_2y,baseline_hba1c_missing,egfr_missing,hdl_missing,total_cholesterol_missing,alt_missing,bmi_final_missing,weight_lb_missing,height_in_missing
0,#A##B,GLP1,2019-07-03,event_0000001,74.0,M,2019-03-28,2019-06-19,LOINC,17856-6,8.0,%,14,97,33.0,45.0000,42.00,141.0,NaN,67.0,NaN,NaN,NaN,67.0,NaN,NaN,2.0,5.0,5.0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0
1,#A##B,SGLT2,2023-12-14,event_0000002,78.0,M,2019-03-28,2023-12-04,LOINC,17856-6,7.5,%,10,1722,18.0,22.0000,63.00,145.0,NaN,66.0,NaN,NaN,NaN,66.0,NaN,NaN,6.0,723.0,108.0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0
2,#A#0B,GLP1,2023-08-16,event_0000003,53.0,F,2022-03-18,2023-08-12,LOINC,4548-4,7.2,%,4,516,140.0,100.0000,3.26,246.0,NaN,62.0,NaN,NaN,NaN,62.0,NaN,NaN,42.0,516.0,160.0,1,0,0,1,0,1,0,0,0,0,0,0,1,1,0
3,#A#1B,SGLT2,2015-07-02,event_0000004,73.0,M,2014-10-13,2015-07-02,LOINC,4548-4,8.8,%,0,262,18.0,98.4532,77.00,175.0,NaN,65.0,NaN,NaN,NaN,65.0,NaN,NaN,12.0,160.0,160.0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0
4,#A#2,DPP4,2018-08-02,event_0000005,47.0,F,2018-04-24,2018-08-02,LOINC,4548-4,8.2,%,0,100,51.0,97.1527,45.00,201.0,NaN,62.5,NaN,NaN,NaN,62.5,NaN,NaN,2.0,91.0,91.0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0
5,#A#2,SGLT2,2018-08-16,event_0000006,47.0,F,2018-04-24,2018-08-02,LOINC,4548-4,8.2,%,14,114,51.0,97.1527,45.00,201.0,NaN,62.5,NaN,NaN,NaN,62.5,NaN,NaN,14.0,105.0,14.0,1,0,0,0,1,0,0,0,0,0,0,0,1,1,0
6,#A#2,GLP1,2020-02-24,event_0000007,49.0,F,2018-04-24,2020-02-06,LOINC,4548-4,9.1,%,18,671,36.0,99.0000,49.00,205.0,NaN,61.0,NaN,NaN,NaN,61.0,NaN,NaN,58.0,662.0,18.0,1,0,0,1,1,0,0,0,0,0,0,0,1,1,0
7,#A#9,DPP4,2008-03-10,event_0000008,67.0,M,2005-10-05,2007-11-17,LOINC,4548-4,7.3,%,114,887,22.0,84.1467,76.00,189.0,NaN,65.0,NaN,NaN,NaN,65.0,NaN,NaN,14.0,670.0,56.0,1,0,0,0,0,1,1,0,0,0,0,0,1,1,0
8,#A#AB,GLP1,2022-02-09,event_0000009,29.0,F,2021-12-27,2022-02-01,LOINC,4548-4,9.1,%,8,44,18.0,137.0000,NaN,NaN,NaN,65.0,NaN,NaN,NaN,65.0,NaN,NaN,4.0,44.0,15.0,1,0,0,0,0,0,0,0,0,1,1,0,1,1,0
9,#A#AD,GLP1,2025-11-19,event_0000010,66.0,M,2025-06-12,2025-09-26,LOINC,4548-4,8.5,%,54,160,12.0,101.0000,36.00,127.0,NaN,67.0,NaN,NaN,NaN,67.0,NaN,NaN,792.0,153.0,41.0,0,1,0,1,0,0,0,0,0,0,0,0,1,1,0


## 9. Build Deterministic Patient-Level Train/Validation/Test Splits

These are patient-level splits so repeated initiations from the same patient stay in the same partition.


In [8]:
def hashed_split(patient_id: str) -> str:
    token = hashlib.md5(str(patient_id).encode('utf-8')).hexdigest()[:8]
    value = int(token, 16) / float(16**8)
    if value < 0.70:
        return 'train'
    if value < 0.85:
        return 'validation'
    return 'test'

patient_splits = pd.DataFrame({
    'patient_id': sorted(static_event_modeling_table['patient_id'].astype('string').dropna().unique().tolist())
})
patient_splits['split'] = patient_splits['patient_id'].apply(hashed_split)
patient_splits.to_csv(PATIENT_SPLITS_PATH, index=False)

event_splits = static_event_modeling_table[['event_id', 'patient_id']].merge(patient_splits, on='patient_id', how='left')
event_splits.to_csv(EVENT_SPLITS_PATH, index=False)

print(PATIENT_SPLITS_PATH)
print(EVENT_SPLITS_PATH)
display(patient_splits['split'].value_counts().rename_axis('split').to_frame('patients'))
display(event_splits['split'].value_counts().rename_axis('split').to_frame('events'))


/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/modeling/patient_splits.csv
/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/modeling/event_splits.csv


,patients
split,
train,66528
test,14151
validation,13972


,events
split,
train,84621
test,17901
validation,17702


## 10. Summarize The Event-Aligned Sequence Domains For Modeling


In [9]:
def summarize_aligned_dataset(path: Path, chunksize: int = 250_000) -> dict:
    if not path.exists() or path.stat().st_size == 0:
        return {'rows': 0, 'patients': 0, 'events': 0, 'feature_names': 0, 'size_mb': 0.0}

    total_rows = 0
    patients = set()
    events = set()
    feature_names = set()

    for chunk in pd.read_csv(path, usecols=lambda c: c in {'patient_id', 'event_id', 'feature_name'}, chunksize=chunksize):
        total_rows += len(chunk)
        if 'patient_id' in chunk.columns:
            patients.update(chunk['patient_id'].dropna().astype('string').unique().tolist())
        if 'event_id' in chunk.columns:
            events.update(chunk['event_id'].dropna().astype('string').unique().tolist())
        if 'feature_name' in chunk.columns:
            feature_names.update(chunk['feature_name'].dropna().astype('string').unique().tolist())

    return {
        'rows': int(total_rows),
        'patients': int(len(patients)),
        'events': int(len(events)),
        'feature_names': int(len(feature_names)),
        'size_mb': round(path.stat().st_size / (1024 * 1024), 2),
    }

sequence_domain_summary = pd.DataFrame([
    {'domain': domain, 'path': str(path), **summarize_aligned_dataset(path)}
    for domain, path in ALIGNED_SEQUENCE_PATHS.items()
])
sequence_domain_summary.to_csv(SEQUENCE_DOMAIN_SUMMARY_PATH, index=False)
display(sequence_domain_summary)


,domain,path,rows,patients,events,feature_names,size_mb
0,sequence_hba1c,/uufs/chpc.utah.edu/common/home/kukhareva-grou...,458493,92201,116919,1,39.77
1,sequence_core_labs,/uufs/chpc.utah.edu/common/home/kukhareva-grou...,3113627,91407,115835,4,326.94
2,sequence_glucose_meds,/uufs/chpc.utah.edu/common/home/kukhareva-grou...,6007280,89319,112862,1,821.47
3,sequence_body_size,/uufs/chpc.utah.edu/common/home/kukhareva-grou...,19588065,83987,107393,3,2305.36


## 11. Outcome Integration Hook

Training the final regression models requires full-project follow-up HbA1c outcome labels. This section checks for an outcome file under `data/full_project/outcomes/` and merges it if found.

Expected minimum columns:

- either `event_id` plus the outcome column
- or `patient_id`, `drug_class`, `index_date` plus the outcome column

Accepted outcome column names in this notebook:

- `outcome_hba1c`
- `closest_outcome_hba1c`
- `followup_hba1c`
- `y`


In [13]:
outcome_path = next((p for p in OUTCOME_CANDIDATE_PATHS if p.exists()), None)

if outcome_path is None:
    outcome_ready_modeling_table = pd.DataFrame()
    print('No full-project outcome label file found yet.')
    print('Predictor datasets and splits are ready. Add one of these files later:')
    for p in OUTCOME_CANDIDATE_PATHS:
        print('-', p)
else:
    outcome_df = pd.read_csv(outcome_path, parse_dates=['index_date'] if 'index_date' in pd.read_csv(outcome_path, nrows=0).columns else None)
    outcome_col_candidates = ['outcome_hba1c', 'closest_outcome_hba1c', 'followup_hba1c', 'y']
    outcome_col = next((c for c in outcome_col_candidates if c in outcome_df.columns), None)

    if outcome_col is None:
        raise ValueError(f'No recognized outcome column found in {outcome_path}. Expected one of: {outcome_col_candidates}')

    if 'event_id' in outcome_df.columns:
        outcome_ready_modeling_table = static_event_modeling_table.merge(
            outcome_df[['event_id', outcome_col]].rename(columns={outcome_col: 'outcome_hba1c'}),
            on='event_id',
            how='inner'
        )
    else:
        required_keys = {'patient_id', 'drug_class', 'index_date'}
        if not required_keys.issubset(set(outcome_df.columns)):
            raise ValueError(f'Outcome file {outcome_path} must contain event_id or patient_id/drug_class/index_date.')
        outcome_df['patient_id'] = outcome_df['patient_id'].astype('string')
        outcome_df['index_date'] = pd.to_datetime(outcome_df['index_date'], errors='coerce')
        outcome_ready_modeling_table = static_event_modeling_table.merge(
            outcome_df[['patient_id', 'drug_class', 'index_date', outcome_col]].rename(columns={outcome_col: 'outcome_hba1c'}),
            on=['patient_id', 'drug_class', 'index_date'],
            how='inner'
        )

    print('Loaded outcome file:', outcome_path)
    print('Outcome-ready rows:', len(outcome_ready_modeling_table))
    print('Outcome-ready patients:', outcome_ready_modeling_table['patient_id'].nunique())
    display(outcome_ready_modeling_table[['event_id', 'patient_id', 'drug_class', 'index_date', 'outcome_hba1c']].head(10))


Loaded outcome file: /uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/outcomes/full_project_outcome_hba1c.csv
Outcome-ready rows: 97673
Outcome-ready patients: 77016


,event_id,patient_id,drug_class,index_date,outcome_hba1c
0,event_0000001,#A##B,GLP1,2019-07-03,6.7
1,event_0000002,#A##B,SGLT2,2023-12-14,7.0
2,event_0000003,#A#0B,GLP1,2023-08-16,6.4
3,event_0000004,#A#1B,SGLT2,2015-07-02,8.1
4,event_0000005,#A#2,DPP4,2018-08-02,6.1
5,event_0000008,#A#9,DPP4,2008-03-10,6.6
6,event_0000009,#A#AB,GLP1,2022-02-09,6.3
7,event_0000011,#A#CB,GLP1,2023-02-07,7.8
8,event_0000012,#A#CB,SGLT2,2024-01-11,7.8
9,event_0000014,#A#DB,DPP4,2016-08-03,9.0


## 12. Static And Sequence Modeling Hand-Off

This section does not force model fitting. Instead it reports whether the notebook is already fully train-ready for the four planned model families:

- Elastic Net
- HistGradientBoostingRegressor
- MLP on static features
- GRU on aligned sequence datasets


In [14]:
static_feature_candidates = [
    'age_at_index',
    'sex',
    'drug_class',
    'baseline_hba1c',
    't2d_duration_days',
    'egfr',
    'hdl',
    'total_cholesterol',
    'alt',
    'bmi_final',
    'weight_lb',
    'height_in',
    'glucose_med_record_rows_2y',
    'first_glucose_med_days_before_index',
    'last_glucose_med_days_before_index',
    'prior_metformin_2y',
    'prior_insulin_2y',
    'prior_glp1_2y',
    'prior_sglt2_2y',
    'prior_dpp4_2y',
    'prior_sulfonylurea_2y',
    'prior_tzd_2y',
]
static_feature_candidates = [c for c in static_feature_candidates if c in static_event_modeling_table.columns]

model_handoff_summary = pd.DataFrame([
    {
        'model_family': 'elastic_net',
        'predictors_ready': len(static_feature_candidates) > 0,
        'outcome_ready': not outcome_ready_modeling_table.empty,
        'note': 'Uses the static event modeling table and patient-level splits.',
    },
    {
        'model_family': 'hist_gradient_boosting',
        'predictors_ready': len(static_feature_candidates) > 0,
        'outcome_ready': not outcome_ready_modeling_table.empty,
        'note': 'Uses the same static table as the Elastic Net baseline.',
    },
    {
        'model_family': 'mlp_static',
        'predictors_ready': len(static_feature_candidates) > 0,
        'outcome_ready': not outcome_ready_modeling_table.empty,
        'note': 'Uses the static event table after numeric/categorical preprocessing.',
    },
    {
        'model_family': 'gru_sequence',
        'predictors_ready': all(path.exists() for path in ALIGNED_SEQUENCE_PATHS.values()),
        'outcome_ready': not outcome_ready_modeling_table.empty,
        'note': 'Uses the aligned sequence files plus the event and patient split files.',
    },
])

display(model_handoff_summary)
print('Static event table:', STATIC_EVENT_TABLE_PATH)
print('Patient splits:', PATIENT_SPLITS_PATH)
print('Event splits:', EVENT_SPLITS_PATH)
print('Static feature summary:', STATIC_FEATURE_SUMMARY_PATH)
print('Sequence domain summary:', SEQUENCE_DOMAIN_SUMMARY_PATH)


,model_family,predictors_ready,outcome_ready,note
0,elastic_net,True,True,Uses the static event modeling table and patie...
1,hist_gradient_boosting,True,True,Uses the same static table as the Elastic Net ...
2,mlp_static,True,True,Uses the static event table after numeric/cate...
3,gru_sequence,True,True,Uses the aligned sequence files plus the event...


Static event table: /uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/modeling/static_event_modeling_table.csv
Patient splits: /uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/modeling/patient_splits.csv
Event splits: /uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/modeling/event_splits.csv
Static feature summary: /uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/modeling/static_feature_summary.csv
Sequence domain summary: /uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/modeling/sequence_domain_summary.csv


## 13. Save A Modeling Manifest


In [15]:
manifest = {
    'static_event_table': str(STATIC_EVENT_TABLE_PATH),
    'patient_splits': str(PATIENT_SPLITS_PATH),
    'event_splits': str(EVENT_SPLITS_PATH),
    'static_feature_summary': str(STATIC_FEATURE_SUMMARY_PATH),
    'sequence_domain_summary': str(SEQUENCE_DOMAIN_SUMMARY_PATH),
    'aligned_sequence_paths': {k: str(v) for k, v in ALIGNED_SEQUENCE_PATHS.items()},
    'outcome_file_detected': str(outcome_path) if 'outcome_path' in globals() and outcome_path is not None else None,
    'static_feature_candidates': static_feature_candidates,
}
MODELING_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
print(MODELING_MANIFEST_PATH)
print('Notebook 11 predictor-ready artifacts are saved.')


/uufs/chpc.utah.edu/common/home/kukhareva-group1/Diabetes_Study/project_Andre/data/full_project/modeling/modeling_manifest.json
Notebook 11 predictor-ready artifacts are saved.


## 14. What Notebook 11 Delivers

After running this notebook, you will have:

- a static event-level modeling table for the non-DL baselines and the static MLP
- deterministic patient-level and event-level split files
- verified aligned sequence domain summaries for the GRU/LSTM track
- a manifest that points to all modeling artifacts on disk

If and when a full-project outcome label file is added under `data/full_project/outcomes/`, the same notebook will merge it and become training-ready for the full analysis.
